<h3>실습 안내</h3><p>문서 ID와 쪽 번호가 있는 문서를 검색합니다. 출처 태그, 모델이 참고할 문맥, 모델에 보낼 프롬프트, 모델 응답 객체인 AIMessage를 차례로 확인합니다.</p>
<h3>검색 결과가 답과 출처가 되는 과정</h3><p>검색된 <code>Document</code>는 모델에 자동 전달되지 않습니다. 코드가 <code>page_content</code>와 <code>metadata</code>를 문맥 문자열로 만든 뒤 <code>model.invoke(prompt)</code>를 호출할 때 생성이 실행됩니다. 반환 객체의 <code>text</code>가 답 문자열입니다.</p><p>프롬프트에 출처를 넣는 것, 답에 출처가 표시되는 것, 표시한 출처가 실제 주장을 뒷받침하는 것은 서로 다른 검사입니다. 이 절의 ID 포함 검사는 문장 단위 사실 검증을 대신하지 않습니다.</p>

In [1]:
# 문서, 벡터 검색, OpenAI 임베딩·생성에 필요한 패키지를 설치합니다.
# ==로 버전을 고정해 실행 환경 차이를 줄입니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3"

Note: you may need to restart the kernel to use updated packages.


<h3>1. API 키 준비</h3><p><code>.env</code>를 로드하고 OpenAI API 키가 존재하는지 확인합니다.</p>

In [2]:
# find_dotenv()는 .env 경로를, os.getenv()는 로드된 API 키 값을 반환합니다.
# assert는 외부 호출 전에 준비 상태를 검사합니다.
import os
from dotenv import find_dotenv, load_dotenv

env_file = find_dotenv(usecwd=True)
assert env_file, ".env를 찾지 못했습니다."
load_dotenv(env_file)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 없습니다."
print("API 키 준비 완료")

API 키 준비 완료


<h3>2. 출처가 있는 문서를 색인</h3><p>각 Document에 문서 ID(<code>doc_id</code>)와 쪽 번호(<code>page</code>)를 넣습니다. 그런 다음 문서를 숫자 벡터로 바꿔 검색할 수 있는 메모리 저장소를 만듭니다.</p>

In [3]:
# documents는 본문과 출처 메타데이터를 가진 Document 리스트입니다.
# from_documents()는 문서를 임베딩해 store를 반환합니다.
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

documents = [
    Document(page_content="정기 휴관일은 매주 월요일입니다.", metadata={"doc_id": "D01", "page": 1}),
    Document(page_content="대출 기간은 14일입니다.", metadata={"doc_id": "D02", "page": 2}),
]
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
store = InMemoryVectorStore.from_documents(documents, embedding)
print("문서 수:", len(documents), "store:", type(store).__name__)

문서 수: 2 store: InMemoryVectorStore


<h3>3. 관련 문서 한 건 검색</h3><p><code>k=1</code>로 가장 가까운 Document 한 건을 가져옵니다. 문서 ID와 쪽 번호를 담은 출처 정보(metadata), 본문을 따로 출력합니다.</p>

In [4]:
# hits는 질문과 가장 관련 높은 Document 한 건을 담은 리스트입니다.
# hit는 hits의 첫 번째 Document를 가리킵니다.
question = "정기 휴관일은 언제인가요?"
hits = store.similarity_search(question, k=1)
hit = hits[0]
print("Document 타입:", type(hit).__name__)
print("metadata:", hit.metadata)
print("content:", hit.page_content)

Document 타입: Document
metadata: {'doc_id': 'D01', 'page': 1}
content: 정기 휴관일은 매주 월요일입니다.


<h3>4. 출처 태그와 문맥 만들기</h3><p>검색한 Document의 문서 ID와 쪽 번호로 <code>[D01 p.1]</code> 모양의 출처 태그를 만들고 본문 앞에 붙입니다.</p>

In [5]:
# source_tag는 [문서ID p.쪽] 형식의 문자열입니다.
# f-string은 중괄호 안의 metadata 값과 page_content를 context에 삽입합니다.
doc_id = hit.metadata["doc_id"]
page = hit.metadata["page"]
source_tag = f"[{doc_id} p.{page}]"
context = f"{source_tag} {hit.page_content}"
print("출처:", source_tag)
print("문맥:", context)

출처: [D01 p.1]
문맥: [D01 p.1] 정기 휴관일은 매주 월요일입니다.


<h3>5. 삼중따옴표로 프롬프트 조립</h3><p>여러 줄 문자열에 문맥과 질문을 삽입하고 모델에 전달되는 전체 프롬프트를 먼저 출력합니다.</p>

In [6]:
# f 삼중 문자열은 여러 줄을 유지하며 context와 question 값을 삽입합니다.
# 생성 전에 prompt를 출력해 모델이 실제로 받는 입력을 확인합니다.
prompt = f"""다음 문서만 근거로 답하세요.
답 끝에 문서의 출처 태그를 그대로 붙이세요.

[문서]
{context}

[질문]
{question}
"""
print(prompt)

다음 문서만 근거로 답하세요.
답 끝에 문서의 출처 태그를 그대로 붙이세요.

[문서]
[D01 p.1] 정기 휴관일은 매주 월요일입니다.

[질문]
정기 휴관일은 언제인가요?



<h3>6. 생성 모델 호출</h3><p><code>init_chat_model</code>로 모델을 만들고 <code>invoke()</code>로 프롬프트를 전달합니다.</p>

In [7]:
# model은 LangChain이 제공하는 OpenAI 대화 모델 객체입니다.
# answer는 AIMessage이고 .text는 생성된 답변 문자열입니다.
from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-5.4-mini", temperature=0)
answer = model.invoke(prompt)
print("answer 타입:", type(answer).__name__)
print(answer.text)
assert "월요일" in answer.text
assert source_tag in answer.text

answer 타입: AIMessage
정기 휴관일은 매주 월요일입니다. [D01 p.1]


<h3>7. k만 2로 바꾸어 문맥 비교</h3><p>질문과 모델은 유지하고 검색 개수만 늘려 프롬프트에 들어갈 후보가 어떻게 늘어나는지 확인합니다.</p>

In [8]:
# more_hits는 k=2로 가져온 Document 리스트입니다.
# ID와 본문 길이를 모두 출력해 k=1 결과와 비교합니다.
more_hits = store.similarity_search(question, k=2)
print("k=1 문서 수:", len(hits))
print("k=2 문서 수:", len(more_hits))
for document in more_hits:
    print(document.metadata["doc_id"], len(document.page_content))
assert len(more_hits) == 2

k=1 문서 수: 1
k=2 문서 수: 2
D01 18
D02 14


<h3>정리</h3><p>Document의 문서 ID와 쪽 번호로 출처 태그를 만들었습니다. 태그와 본문을 프롬프트에 넣었고, <code>invoke()</code>가 반환한 AIMessage에서 답과 출처를 읽었습니다.</p>

<p><strong>원인</strong> — 검색 결과를 새 Document로 옮기면서 <code>page</code>를 빠뜨리면 쪽 번호가 있는 출처 태그를 만들 수 없습니다.</p>

In [9]:
# broken은 실제 검색 결과 hit의 본문과 문서 ID만 옮겨 page를 빠뜨린 경우입니다.
# metadata["page"]를 읽을 때 KeyError가 발생하는지 확인합니다.
broken = Document(
    page_content=hit.page_content,
    metadata={"doc_id": hit.metadata["doc_id"]},
)
try:
    print(f"[{broken.metadata['doc_id']} p.{broken.metadata['page']}]")
except KeyError as error:
    print("예상한 실패: 누락된 메타데이터", error)

예상한 실패: 누락된 메타데이터 'page'


<p><strong>수정</strong> — 같은 검색 결과에서 실제 <code>page</code> 값도 함께 옮깁니다. 복구한 출처 태그와 본문으로 같은 질문의 프롬프트를 만들고 실제 답까지 확인합니다.</p>

In [10]:
# fixed는 hit에 있던 실제 doc_id와 page를 모두 복사한 Document입니다.
# 복구한 태그와 본문으로 같은 질문을 다시 실행해 출처가 답에 남는지 확인합니다.
fixed = Document(
    page_content=hit.page_content,
    metadata={"doc_id": hit.metadata["doc_id"], "page": hit.metadata["page"]},
)
fixed_tag = f"[{fixed.metadata['doc_id']} p.{fixed.metadata['page']}]"
fixed_context = f"{fixed_tag} {fixed.page_content}"
fixed_prompt = f"""다음 문서만 근거로 답하세요. 답 끝에 출처 태그를 붙이세요.

[문서]
{fixed_context}

[질문]
{question}"""
fixed_answer = model.invoke(fixed_prompt)
print("After 태그:", fixed_tag)
print("After 답:", fixed_answer.text)
assert fixed_tag == source_tag
assert fixed_tag in fixed_answer.text

After 태그: [D01 p.1]
After 답: 정기 휴관일은 매주 월요일입니다. [D01 p.1]
